# 🛡️ Deepfake Detection — GPU Training on Google Colab

### Steps:
1. ✅ Connect to GPU
2. ✅ Mount Google Drive
3. ✅ Upload & extract your dataset
4. ✅ Augment Real images to balance dataset (2966 → 5000)
5. ✅ Train CNN model with GPU
6. ✅ Download trained model

> **Before running:** Upload your `dataset.zip` to Google Drive first.

In [ ]:
# ============================================================
# CELL 1 — Check GPU
# ============================================================
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('🔥 GPUs Available:', gpus)

if gpus:
    print('✅ GPU is ready! Training will be FAST.')
else:
    print('⚠️ No GPU found. Go to Runtime → Change Runtime Type → GPU')

In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
# ============================================================
# CELL 3 — Upload dataset.zip from Google Drive
# ============================================================
# CHANGE THIS PATH to where your dataset.zip is in your Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/dataset.zip'

import shutil, os

# Copy zip to Colab working directory
shutil.copy(DRIVE_ZIP_PATH, '/content/dataset.zip')
print('✅ dataset.zip copied from Drive')

# Extract it
import zipfile
with zipfile.ZipFile('/content/dataset.zip', 'r') as z:
    z.extractall('/content/')

# Count images
real_count = len(os.listdir('/content/dataset/real'))
fake_count = len(os.listdir('/content/dataset/fake'))
print(f'📁 Before augmentation — Real: {real_count}, Fake: {fake_count}')

In [ ]:
# ============================================================
# CELL 4 — Augment Real Images to Balance Dataset (target: 5000)
# ============================================================
import cv2
import numpy as np
import random
import os

REAL_DIR = '/content/dataset/real'
TARGET = 5000

def augment_image(img):
    """Apply random augmentations to an image."""
    # Horizontal flip (50% chance)
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    # Brightness adjustment
    factor = random.uniform(0.75, 1.25)
    img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    # Contrast adjustment
    alpha = random.uniform(0.8, 1.2)
    img = np.clip(alpha * img.astype(np.float32), 0, 255).astype(np.uint8)

    # Random rotation ±15 degrees
    angle = random.uniform(-15, 15)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    return img


existing_files = [f for f in os.listdir(REAL_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
current_count = len(existing_files)
needed = TARGET - current_count

print(f'🔧 Real images now: {current_count} | Need to generate: {needed} more')

aug_counter = 0
pool = existing_files.copy()

while aug_counter < needed:
    # Pick a random source image
    src_name = random.choice(pool)
    src_path = os.path.join(REAL_DIR, src_name)
    img = cv2.imread(src_path)

    if img is None:
        continue

    img_aug = augment_image(img)

    # Save with unique name
    base = os.path.splitext(src_name)[0]
    out_name = f'{base}_aug_{aug_counter}.jpg'
    cv2.imwrite(os.path.join(REAL_DIR, out_name), img_aug)

    aug_counter += 1

    if aug_counter % 500 == 0:
        print(f'  → Generated {aug_counter}/{needed} augmented images...')

final_real = len(os.listdir(REAL_DIR))
final_fake = len(os.listdir('/content/dataset/fake'))
print(f'\n✅ After augmentation — Real: {final_real}, Fake: {final_fake}')
print('Dataset is now BALANCED! 🎉')

In [ ]:
# ============================================================
# CELL 5 — Build Improved CNN Model
# ============================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)

def create_model():
    model = Sequential([
        # Normalize pixel values
        tf.keras.layers.Rescaling(1./255, input_shape=(224, 224, 3)),

        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Flatten(),

        # Fully connected
        Dense(256, activation='relu'),
        Dropout(0.4),   # Prevents overfitting
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')  # Output: 0=Fake, 1=Real
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_model()
model.summary()
print('\n✅ Model built successfully!')

In [ ]:
# ============================================================
# CELL 6 — Load Dataset with Validation Split
# ============================================================
import tensorflow as tf

BATCH_SIZE = 32   # Larger batch = faster GPU training
IMAGE_SIZE = (224, 224)
DATASET_DIR = '/content/dataset'

# Training data (80%)
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Validation data (20%)
val_data = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# Optimize I/O with prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(buffer_size=AUTOTUNE)
val_data   = val_data.prefetch(buffer_size=AUTOTUNE)

print('✅ Dataset loaded!')
print(f'   Class names: {train_data.class_names}')  # ["fake", "real"]
print(f'   Train batches: {len(train_data)}')
print(f'   Val batches:   {len(val_data)}')

In [ ]:
# ============================================================
# CELL 7 — Train Model (GPU Accelerated)
# ============================================================
import os

os.makedirs('/content/model', exist_ok=True)

callbacks = [
    # Save BEST model automatically (based on val_accuracy)
    tf.keras.callbacks.ModelCheckpoint(
        filepath='/content/model/deepfake_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    # Stop early if no improvement for 5 epochs (saves time)
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate if stuck
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )
]

print('🚀 Starting GPU training...')
print('=' * 50)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks
)

print('\n✅ Training complete!')
print(f'Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')

In [ ]:
# ============================================================
# CELL 8 — Plot Training Results
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='royalblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'],     label='Train Loss', color='royalblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_results.png', dpi=150)
plt.show()
print('✅ Training graph saved!')

In [ ]:
# ============================================================
# CELL 9 — Save Model to Google Drive
# ============================================================
import shutil, os

# Save to Drive so you don't lose it
DRIVE_SAVE_PATH = '/content/drive/MyDrive/deepfake_model.h5'
shutil.copy('/content/model/deepfake_model.h5', DRIVE_SAVE_PATH)

model_size = os.path.getsize(DRIVE_SAVE_PATH) / (1024*1024)
print(f'✅ Model saved to Google Drive!')
print(f'   Path: {DRIVE_SAVE_PATH}')
print(f'   Size: {model_size:.1f} MB')

In [ ]:
# ============================================================
# CELL 10 — Download Model to Your Computer
# ============================================================
from google.colab import files

print('📥 Starting download of deepfake_model.h5...')
files.download('/content/model/deepfake_model.h5')
print('✅ Download started! Save it to: deepfake_project/model/deepfake_model.h5')